# 实验7：X86 + CANN 异构指令系统认知与执行流程验证

## 一、实验说明

### 1.1 实验背景
随着人工智能应用的爆发，传统以 **X86 CPU** 为核心的通用计算系统在处理大规模并行计算任务（如矩阵乘、卷积）时面临性能与能效瓶颈。**昇腾AI处理器**内部集成大量计算核心（AI Core），通过专用的 **达芬奇架构指令集** 高效执行向量、矩阵等智算操作，而CPU则专注于控制逻辑、任务调度与数据预处理。这种 **“CPU通算 + NPU智算”** 的异构组合已成为现代AI计算的主流范式。

为屏蔽硬件细节、统一编程体验，华为提供了 **CANN（Compute Architecture for Neural Networks）** 异构计算架构。其中：
- **X86通用指令**（如`mov`、`call`、`syscall`）运行在CPU侧，负责流程控制、任务队列管理、驱动调用；
- **CANN加速指令** 经过TBE/ASCEND C算子开发框架编译后，生成面向AI Core的专用指令序列，在NPU上执行矩阵运算、向量激活等密集型操作。

本实验基于CANNLab一站式开发平台或配置有CANN Toolkit的云环境，通过一个完整的“CPU侧ACL调度 + NPU侧算子执行”流程，使同学们直观理解两类指令的分工与协同，掌握 **AC（Ascend Computing）基础框架**、**核函数/算子调度**等核心技能。


### 1.2 实验环境准备（**CANNLab在线实验跳过这一环节**）
- **硬件**：沙箱提供X86_64 CPU + 昇腾NPU（如Atlas 300I推理卡 / 910训练卡虚拟实例）。
- **操作系统**：Ubuntu 18.04 / 20.04。
- **软件栈**：CANN 5.0.4 及以上（含昇腾驱动、固件、toolkit、ACL库、TBE算子编译器等）。
- **环境变量**：登录后执行以下命令使CANN生效。
  ```bash
  source /usr/local/Ascend/ascend-toolkit/set_env.sh
  ```
- **验证环境**：
  ```bash
  npu-smi info        # 查看NPU状态
  lscpu               # 查看CPU信息（确认x86_64架构）
  ```

## 二、实验任务

### 2.1 任务描述
编写一个异构计算程序，调用 **ACL（Ascend Computing Language）** 接口在CPU侧下发 **矩阵乘（GEMM）** 任务，由NPU执行计算。通过**反汇编分析、profiling跟踪、多流并发调度**等环节，完成以下任务：
1. 观察X86侧负责控制流与任务下发的汇编指令；
2. 分析NPU侧执行算子时的异步执行流程；
3. 使用昇腾Profiling工具抓取任务调度时间线，验证CPU与NPU的协同；
4. 通过多流并发实验，理解算子调度与并行机制。

### 2.2 学习目标
- 理解X86通用指令与CANN加速指令的职责边界；
- 掌握ACL基础编程框架（初始化、流管理、算子执行、同步）；
- 熟悉昇腾Profiling工具（msprof）的使用与Timeline分析；
- 具备算子调度、多流并发的设计能力，为后续核函数开发奠定基础。

## 三、任务准备

### 3.1 前置知识
1. **X86-64指令集基础**：了解常用指令格式（`mov`, `call`, `ret`, `syscall`等），能够使用`objdump`反汇编二进制文件。
2. **CANN架构与ACL编程**：
   - **ACL**：提供设备管理、上下文、流、内存、算子的C++ API。
   - **Stream（流）**：任务队列，算子下发后异步执行，通过`aclrtSynchronizeStream`同步。
   - **TBE/Ascend C**：自定义算子开发方式，最终编译为NPU可执行指令。
3. **异构调度原理**：CPU负责将算子描述、输入输出地址打包成Task，通过驱动提交到NPU的任务调度器（TS），NPU上的AI Core取指执行。

### 3.2 实验数据准备
采用随机生成的`1024×1024`矩阵，数据类型为`float16`（适应NPU Tensor Core高效计算）。实验中将在CPU侧分配Host/Device内存，并将数据拷贝至NPU。

## 四、任务实施


### 步骤1：准备GEMM算子的模型文件

构造GEMM算子的描述文件（gemm.json文件，描述输入输出Tensor描述、算子属性等）。 

```json
[
{
  "op": "GEMM",
  "input_desc": [
    {
      "format": "ND",
      "shape": [1024, 1024],
      "type": "float16"
    },
    {
      "format": "ND",
      "shape": [1024, 1024],
      "type": "float16"
    },
    {
      "format": "ND",
      "shape": [1024, 1024],
      "type": "float16"
    },
    {
      "format": "ND",
      "shape": [],
      "type": "float16"
    },
    {
      "format": "ND",
      "shape": [],
      "type": "float16"
    }
  ],
  "output_desc": [
    {
      "format": "ND",
      "shape": [1024, 1024],
      "type": "float16"
    }
  ],
  "attr": [
  {
    "name": "transpose_a",
    "type": "bool",
    "value": false
  },
  {
    "name": "transpose_b",
    "type": "bool",
    "value": false
    }
  ]
}
]
```

借助ATC工具，将该算子描述文件编译成单算子模型文件（*.om文件），再分别调用AscendCL接口加载om模型文件、执行算子。 

```bash
atc --singleop=$HOME/experiment/lab_07/gemm.json --output=$HOME/experiment/lab_07/op_model --soc_version=<soc_version>
```

In [1]:
!atc --singleop=$HOME/experiment/lab_07/gemm.json --output=$HOME/experiment/lab_07/op_model --soc_version=Ascend910B3

ATC start working now, please wait for a moment.
....
ATC run success, welcome to the next use.



### 步骤2：编写CPU侧调度程序
在 `/home/developer/experiment/lab_07/src/` 目录下创建`main.cpp`，使用ACL的`aclblasGemmEx`接口下发矩阵乘。该接口内部将GEMM算子封装为NPU可执行任务，代码框架如下：

```cpp
#include <iostream>
#include <cstdlib>
#include <cstring>
#include <ctime>
#include <cmath>
#include "acl/acl.h"
#include "acl/ops/acl_cblas.h"

#define M 1024
#define N 1024
#define K 1024

// 检查ACL状态
#define CHECK_ACL(ret, msg) \
    do { \
        if (ret != ACL_SUCCESS) { \
            std::cerr << msg << " failed with error: " << ret << std::endl; \
            return ret; \
        } \
    } while(0)

// 初始化矩阵数据
void initMatrix(aclFloat16* mat, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        mat[i] = (aclFloat16)(rand() % 100) / 10.0f;
    }
}

// CPU矩阵乘验证
void cpuGemm(const aclFloat16* A, const aclFloat16* B, aclFloat16* C, int m, int n, int k) {
    memset(C, 0, m * n * sizeof(aclFloat16));
    for (int i = 0; i < m; i++) {
        for (int j = 0; j < n; j++) {
            aclFloat16 sum = 0.0f;
            for (int l = 0; l < k; l++) {
                sum += A[i * k + l] * B[l * n + j];
            }
            C[i * n + j] = sum;
        }
    }
}

bool verifyResult(const aclFloat16* npuC, const aclFloat16* cpuC, int m, int n) {
    aclFloat16 maxDiff = 0.0f;
    for (int i = 0; i < m * n; i++) {
        aclFloat16 diff = std::abs(npuC[i] - cpuC[i]);
        if (diff > maxDiff) maxDiff = diff;
    }
    std::cout << "Max difference: " << maxDiff << std::endl;
    return maxDiff < 1e-4f;  // 放宽阈值以适应NPU计算精度
}

int main() {
    std::cout << "=== ACL GEMM Example ===" << std::endl;
    std::cout << "Matrix dimensions: M=" << M << ", N=" << N << ", K=" << K << std::endl;
    
    srand(static_cast<unsigned>(time(nullptr)));
    
    // 初始化ACL
    aclError ret = aclInit(nullptr);
    CHECK_ACL(ret, "aclInit");
    
    // 设置设备
    ret = aclrtSetDevice(0);
    CHECK_ACL(ret, "aclrtSetDevice");
    
    // 创建Stream
    aclrtStream stream;
    ret = aclrtCreateStream(&stream);
    CHECK_ACL(ret, "aclrtCreateStream");

    ret = aclopSetModelDir("../op_model");
    CHECK_ACL(ret, "aclopSetModelDir");

    // 矩阵维度
    size_t sizeA = M * K * sizeof(aclFloat16);
    size_t sizeB = K * N * sizeof(aclFloat16);
    size_t sizeC = M * N * sizeof(aclFloat16);
    
    // 分配主机内存
    aclFloat16* h_A = new aclFloat16[M * K];
    aclFloat16* h_B = new aclFloat16[K * N];
    aclFloat16* h_C_npu = new aclFloat16[M * N];
    aclFloat16* h_C_cpu = new aclFloat16[M * N];
    
    // 初始化数据
    initMatrix(h_A, M, K);
    initMatrix(h_B, K, N);
    memset(h_C_npu, 0, sizeC);
    
    // 分配设备内存
    void* d_A = nullptr;
    void* d_B = nullptr;
    void* d_C = nullptr;
    ret = aclrtMalloc(&d_A, sizeA, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_ACL(ret, "aclrtMalloc d_A");
    ret = aclrtMalloc(&d_B, sizeB, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_ACL(ret, "aclrtMalloc d_B");
    ret = aclrtMalloc(&d_C, sizeC, ACL_MEM_MALLOC_HUGE_FIRST);
    CHECK_ACL(ret, "aclrtMalloc d_C");
    
    // 拷贝数据到设备
    ret = aclrtMemcpy(d_A, sizeA, h_A, sizeA, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_ACL(ret, "aclrtMemcpy d_A");
    ret = aclrtMemcpy(d_B, sizeB, h_B, sizeB, ACL_MEMCPY_HOST_TO_DEVICE);
    CHECK_ACL(ret, "aclrtMemcpy d_B");
    
    // 参数设置
    aclFloat16 alpha = 1.0f;
    aclFloat16 beta = 0.0f;
    
    std::cout << "\nExecuting aclblasGemmEx on NPU..." << std::endl;
    
    // aclblasGemmEx 调用
    ret = aclblasGemmEx(
        ACL_TRANS_N,          // transA: 矩阵A不转置
        ACL_TRANS_N,          // transB: 矩阵B不转置
        ACL_TRANS_N,          // transC: 当前仅支持ACL_TRANS_N[citation:1][citation:2]
        M,                    // m: A的行数, C的行数
        N,                    // n: B的列数, C的列数
        K,                    // k: A的列数, B的行数
        &alpha,               // alpha指针
        d_A,                  // 矩阵A
        -1,                   // lda: 预留参数，当前只能设置为-1[citation:1][citation:2]
        ACL_FLOAT16,            // dataTypeA: 使用ACL_FLOAT枚举值[citation:10]
        d_B,                  // 矩阵B
        -1,                   // ldb: 预留参数，当前只能设置为-1[citation:1][citation:2]
        ACL_FLOAT16,            // dataTypeB: 使用ACL_FLOAT枚举值[citation:10]
        &beta,                // beta指针
        d_C,                  // 矩阵C
        -1,                   // ldc: 预留参数，当前只能设置为-1[citation:1][citation:2]
        ACL_FLOAT16,            // dataTypeC: 使用ACL_FLOAT枚举值[citation:10]
        ACL_COMPUTE_HIGH_PRECISION,  // type: 计算精度
        stream                // stream
    );
    
    CHECK_ACL(ret, "aclblasGemmEx");
    
    // 等待计算完成
    //ret = aclrtSynchronizeStream(stream);
    ret = aclrtSynchronizeStream(nullptr);
    CHECK_ACL(ret, "aclrtSynchronizeStream");
    
    // 拷贝结果回主机
    ret = aclrtMemcpy(h_C_npu, sizeC, d_C, sizeC, ACL_MEMCPY_DEVICE_TO_HOST);
    CHECK_ACL(ret, "aclrtMemcpy d_C to host");
    
    // CPU验证
    std::cout << "\nComputing on CPU for verification..." << std::endl;
    cpuGemm(h_A, h_B, h_C_cpu, M, N, K);
    
    // 验证结果
    if (verifyResult(h_C_npu, h_C_cpu, M, N)) {
        std::cout << "\n✓ Verification PASSED!" << std::endl;
    } else {
        std::cout << "\n✗ Verification FAILED!" << std::endl;
    }
    
    // 清理资源
    aclrtDestroyStream(stream);
    aclrtFree(d_A);
    aclrtFree(d_B);
    aclrtFree(d_C);
    aclrtResetDevice(0);
    aclFinalize();
    
    delete[] h_A;
    delete[] h_B;
    delete[] h_C_npu;
    delete[] h_C_cpu;
    
    std::cout << "\n=== Done ===" << std::endl;
    return 0;
}
```

编写编译文件`CMakeLists.txt`，它需链接`acl`、`acl_cblas`等库。

```txt
cmake_minimum_required(VERSION 3.10)
project(ACL_GEMM_Example)

set(CMAKE_CXX_STANDARD 11)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

# 设置CANN安装路径（根据实际路径修改）
set(CANN_ROOT "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation root")

# ========== 头文件路径 ==========
# 根据官方文档，头文件在 ${CANN_ROOT}/include/ 目录下[citation:6][citation:9]
include_directories(
    ${CANN_ROOT}/include
)

# ========== 库文件路径 ==========
# 编译时使用stub目录下的库文件，运行时才会链接到实际设备上的库[citation:3][citation:4]
link_directories(
    ${CANN_ROOT}/lib64
    ${CANN_ROOT}/runtime/lib64/stub   # stub库，用于编译链接
)

# ========== 可执行文件 ==========
add_executable(gemm_acl main.cpp)

# ========== 链接库文件 ==========
# 根据官方头文件-库文件对应表[citation:1][citation:6]：
# - acl/acl.h        -> libascendcl.so (链接时用 ascendcl)
# - acl/ops/acl_cblas.h -> libacl_cblas.so (链接时用 acl_cblas)
target_link_libraries(gemm_acl
    ascendcl       # 对应 libascendcl.so，基础ACL库
    acl_cblas      # 对应 libacl_cblas.so，CBLAS库（包含gemm接口）
    pthread
    stdc++
)

# 设置RPATH方便运行时查找库
set_target_properties(gemm_acl PROPERTIES
    INSTALL_RPATH "${CANN_ROOT}/lib64"
    BUILD_WITH_INSTALL_RPATH TRUE
)

# 打印配置信息便于调试
message(STATUS "CANN Root: ${CANN_ROOT}")
message(STATUS "Include path: ${CANN_ROOT}/include")
message(STATUS "Library path: ${CANN_ROOT}/lib64")
```

### 步骤3：X86通用指令认知（反汇编分析）
编译生成可执行文件`gemm_acl`后，用`objdump -d -S gemm_acl`反汇编，定位`main`函数中调用`aclblasGemmEx`的部分：

```assembly
  **1ca4:	48 8d 45 98          	lea    -0x68(%rbp),%rax
    1ca8:	50                   	push   %rax
    1ca9:	41 b9 00 04 00 00    	mov    $0x400,%r9d
    1caf:	41 b8 00 04 00 00    	mov    $0x400,%r8d
    1cb5:	b9 00 04 00 00       	mov    $0x400,%ecx
    1cba:	ba 00 00 00 00       	mov    $0x0,%edx
    1cbf:	be 00 00 00 00       	mov    $0x0,%esi
    1cc4:	bf 00 00 00 00       	mov    $0x0,%edi
    1cc9:	e8 92 f5 ff ff       	call   1260 <aclblasGemmEx@plt>
    1cce:	48 83 c4 70          	add    $0x70,%rsp
    1cd2:	89 45 9c             	mov    %eax,-0x64(%rbp)**
    ...
    1d38:	e8 f3 f4 ff ff       	call   1230 <aclrtSynchronizeStream@plt>
```

**分析**：
- `lea`、`mov`等通用指令负责准备参数并压栈；
- `call aclblasGemmEx` 跳转到ACL库内部实现，该函数进一步调用驱动将**算子Task描述符**写入NPU的命令队列；
- 本质上，CPU执行的“指令”均属x86指令集，其作用是**组织并下发任务**，而非参与实际矩阵运算。

### 步骤4：NPU加速指令执行流程（Profiling验证）
使用昇腾Profiling工具采集运行时数据，分析NPU算子调度与执行指令流。

```bash
# 开启Profiling环境
export PROFILING_MODE=true
export PROFILING_OPTIONS="task_trace:on"

# 使用msprof解析，运行程序gemm_acl，采集结果在 ./profiling_data 目录
msprof --application="./gemm_acl" --output="./profiling_data" --ai-core=on
```

采集完成后，数据目录中会生成关键文件：**msprof_*.json：Timeline时间线文件**，可用Chrome tracing或MindStudio Insight加载，直观展示Host/Device的任务流。

**得出结论**：真正执行**矩阵乘累加指令**（如`vmla.f16`向量乘加）发生在AI Core上，属于CANN加速指令范畴，CPU在此过程中仅做同步等待。

### 步骤5：多流并发与算子调度验证
修改程序，创建两个Stream，分别下发**GEMM**和**Relu**算子（后者使用`aclopExecuteV2`加载自定义TBE算子），实现并发：

```cpp
aclrtStream stream1, stream2;
aclrtCreateStream(&stream1);
aclrtCreateStream(&stream2);

aclblasGemmEx(handle1, ..., stream1);   // Stream1 矩阵乘
aclopExecuteV2(relu_op, ..., stream2); // Stream2 激活算子

aclrtSynchronizeStream(stream1);
aclrtSynchronizeStream(stream2);
```

再次Profiling，Timeline显示两个Task时间重叠，表明**NPU任务调度器（TS）** 能够将不同流的算子指令并行发射到空闲AI Core，体现了CPU仅负责任务排队，真正调度由NPU硬件完成。

## 五、任务拓展

1. **自定义算子指令映射**  
   使用ASCEND C（原TBE）编写一个简单的`Sqrt`算子，通过`ccec`编译器生成NPU kernel文件（`.o`）。执行`readelf -s`可查看导出的kernel symbol，并尝试用昇腾工具链导出二进制段，思考其中指令与达芬奇架构向量指令的对应关系。

2. **CPU与NPU负载分工调优**  
   修改实验代码，将数据预处理（如归一化）放在CPU侧，计算放在NPU侧，在Profiling中标记CPU热点函数（如`normalize_cpu`），观察总执行时间。尝试将部分轻量计算也移至NPU，对比性能，体会“繁重智算下放，轻量逻辑上浮”的设计原则。

3. **深入核函数调度器**  
   阅读昇腾文档中关于Task Scheduler和Runtime框架的描述，画出从`aclopExecuteV2`调用到AI Core取指执行的全链路流程图，标注出X86指令和NPU指令起作用的阶段。


## 六、实验总结

通过本次实验，我们深入验证了 **X86通用指令与CANN加速指令的异构分工**：

- **CPU（X86）侧**：运行ACL框架、用户程序逻辑，通过`call`、`mov`等指令完成算子参数封装、任务下发、同步控制，本质上是一系列服务性操作；
- **NPU侧**：接收到任务描述符后，AI Core从中提取**达芬奇加速指令序列**（向量乘加、矩阵转置、激活函数等），在专用计算单元上高效执行；
- **AC框架**：提供了统一的API接口，屏蔽底层指令差异，实现了通算与智算的无缝协同。

通过反汇编、Profiling及多流调度的实践，同学们已具备初步的**核函数/算子调度**分析能力，为后续定制高性能AI算子、优化异构应用奠定了坚实基础。